**Setup folders and mount Google Drive**

In [9]:
# Cell 1: Setup folders and mount Google Drive
import os
from google.colab import drive
drive.mount('/content/drive')

# !!! ADJUST THIS PATH TO MATCH YOUR DRIVE FOLDER !!!
BASE_PATH = "/content/drive/MyDrive/XMLROBERT/Dataset_Dimension_Comparison"

# Path to your existing CSV splits (train.csv, validation.csv, test.csv)
# Correcting DATA_PATH based on available files in /content/sample_data/
DATA_PATH = "/content/sample_data/datasets"

# Create folders for preprocessed data and results
PREPROCESSED_PATH = os.path.join(BASE_PATH, "preprocessed_dataset")
RESULTS_PATH = os.path.join(BASE_PATH, "results")

os.makedirs(PREPROCESSED_PATH, exist_ok=True)
os.makedirs(RESULTS_PATH, exist_ok=True)

print(f"Base path: {BASE_PATH}")
print(f"Data path: {DATA_PATH}")
print(f"Preprocessed path: {PREPROCESSED_PATH}")
print(f"Results path: {RESULTS_PATH}")
print("\nFolders ready.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Base path: /content/drive/MyDrive/XMLROBERT/Dataset_Dimension_Comparison
Data path: /content/sample_data/datasets
Preprocessed path: /content/drive/MyDrive/XMLROBERT/Dataset_Dimension_Comparison/preprocessed_dataset
Results path: /content/drive/MyDrive/XMLROBERT/Dataset_Dimension_Comparison/results

Folders ready.


**Load and examine split data**

In [10]:
# Cell 2: Load and examine split data
import pandas as pd
import numpy as np
import os
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

# Load pre-split data
train = pd.read_csv(os.path.join(DATA_PATH, "train.csv"))
val = pd.read_csv(os.path.join(DATA_PATH, "validation.csv"))
test = pd.read_csv(os.path.join(DATA_PATH, "test.csv"))

print("Train shape:", train.shape)
print("Val shape:", val.shape)
print("Test shape:", test.shape)
print("\nTrain columns:", train.columns.tolist())
print("\nFirst few rows of train:")
print(train.head(3))

Train shape: (110645, 9)
Val shape: (15806, 9)
Test shape: (31614, 9)

Train columns: ['id', 'Book_Name', 'Writer_Name', 'Category', 'Rating', 'Review', 'Site', 'sentiment', 'label']

First few rows of train:
      id                                          Book_Name  \
0  19635                নেভার স্টপ লার্নিং (হার্ডকভার)        
1  31528   রাসুলুল্লাহ (স) এর নামায (১ম ও ২য় খণ্ড একত্রে...   
2   9235                  পোয়েটিক জাস্টিস (পেপারব্যাক)        

                                   Writer_Name  \
0                                 আয়মান সাদিক    
1   আল্লামা মুহাম্মদ নাসীরুদ্দীন আলবানী (রহঃ)    
2                              আগাথা ক্রিস্টি    

                                            Category  Rating  \
0                                  ছাত্রজীবন উন্নয়ন        5   
1                                       সালাত/নামায        1   
2   রহস্য, গোয়েন্দা, ভৌতিক, মিথ, থ্রিলার, ও অ্যাড...       5   

                                              Review      Site sentiment  \
0  ম

**Check label distribution in splits**

In [11]:
# Cell 3: Check label distribution in splits
TEXT_COL = "Review"        # column containing the review text
LABEL_COL = "label"         # column with numeric labels (0, 2)

def check_distribution(df, name):
    print(f"\n{name} distribution:")
    print(df[LABEL_COL].value_counts().sort_index())
    print(df[LABEL_COL].value_counts(normalize=True).sort_index().round(3))

check_distribution(train, "Train")
check_distribution(val, "Validation")
check_distribution(test, "Test")

# Check split ratio
total = len(train) + len(val) + len(test)
print(f"\nSplit ratios:")
print(f"Train: {len(train)/total:.2%}")
print(f"Val:   {len(val)/total:.2%}")
print(f"Test:  {len(test)/total:.2%}")


Train distribution:
label
0     6772
1     4763
2    99110
Name: count, dtype: int64
label
0    0.061
1    0.043
2    0.896
Name: proportion, dtype: float64

Validation distribution:
label
0      967
1      680
2    14159
Name: count, dtype: int64
label
0    0.061
1    0.043
2    0.896
Name: proportion, dtype: float64

Test distribution:
label
0     1935
1     1361
2    28318
Name: count, dtype: int64
label
0    0.061
1    0.043
2    0.896
Name: proportion, dtype: float64

Split ratios:
Train: 70.00%
Val:   10.00%
Test:  20.00%


**Convert to binary labels and save preprocessed splits**

In [18]:
# Cell 4: Convert to binary labels and save preprocessed splits
import pandas as pd # Ensure pandas is imported
import os # Ensure os is imported

# Reload original data to ensure correct columns are available
train_orig = pd.read_csv(os.path.join(DATA_PATH, "train.csv"))
val_orig = pd.read_csv(os.path.join(DATA_PATH, "validation.csv"))
test_orig = pd.read_csv(os.path.join(DATA_PATH, "test.csv"))

def prepare_data(df):
    """Convert dataframe to binary format: text, label (0/1)"""
    df = df.copy()
    # Keep only needed columns and rename
    df = df[[TEXT_COL, LABEL_COL]].rename(columns={TEXT_COL: "text", LABEL_COL: "label"})

    # Convert to string and drop empty
    df["text"] = df["text"].astype(str)
    df = df.dropna(subset=["text", "label"])
    df = df[df["text"].str.strip().astype(bool)]

    # Keep only 0 and 2, map 2→1, 0→0
    df = df[df["label"].isin([0, 2])]
    df["label"] = df["label"].map({0: 0, 2: 1}).astype(int)

    return df

# Prepare each split using the reloaded original data
train_bin = prepare_data(train_orig)
val_bin = prepare_data(val_orig)
test_bin = prepare_data(test_orig)

print("Train binary shape:", train_bin.shape)
print("Label distribution in train:")
print(train_bin["label"].value_counts())
print("0 = Negative, 1 = Positive")

# Save binary versions in preprocessed folder
train_bin.to_csv(os.path.join(PREPROCESSED_PATH, "BanglaBook_train_binary.csv"), index=False)
val_bin.to_csv(os.path.join(PREPROCESSED_PATH, "BanglaBook_val_binary.csv"), index=False)
test_bin.to_csv(os.path.join(PREPROCESSED_PATH, "BanglaBook_test_binary.csv"), index=False)

print(f"\nBinary splits saved to: {PREPROCESSED_PATH}")

Train binary shape: (105882, 2)
Label distribution in train:
label
1    99110
0     6772
Name: count, dtype: int64
0 = Negative, 1 = Positive

Binary splits saved to: /content/drive/MyDrive/XMLROBERT/Dataset_Dimension_Comparison/preprocessed_dataset


**Zero-shot evaluation with XLM-RoBERTa (honest: neutrals count as wrong)**

In [13]:
# Cell 5: Zero-shot evaluation with XLM-RoBERTa (honest: neutrals count as wrong)
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, DataCollatorWithPadding

MODEL_NAME = "cardiffnlp/twitter-xlm-roberta-base-sentiment"

# Load binary test set
test_df = pd.read_csv(os.path.join(PREPROCESSED_PATH, "BanglaBook_test_binary.csv"))
test_df = test_df.rename(columns={"text": "text", "label": "label"})
test_df["text"] = test_df["text"].astype(str)
test_df["label"] = test_df["label"].astype(int)

# Create HF dataset
test_ds = Dataset.from_pandas(test_df[["text", "label"]])

# Tokenize
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
collator = DataCollatorWithPadding(tokenizer=tokenizer)

def tok(batch):
    return tokenizer(batch["text"], truncation=True, max_length=128)

test_ds = test_ds.map(tok, batched=True)
test_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])

# Load model (3-class: neg/neu/pos)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

# Predict
trainer = Trainer(model=model, data_collator=collator)
pred = trainer.predict(test_ds)

logits = pred.predictions
pred_3class = np.argmax(logits, axis=1)  # 0=neg, 1=neu, 2=pos

# Honest mapping: neutral → wrong answer
y_true = test_df["label"].values
pred_mapped = np.where(pred_3class == 2, 1, 0)          # pos(2)→1, neg(0)→0
pred_honest = pred_mapped.copy()
neutral_mask = (pred_3class == 1)
pred_honest[neutral_mask] = 1 - y_true[neutral_mask]   # force opposite of true

# Metrics
acc = accuracy_score(y_true, pred_honest)
f1 = f1_score(y_true, pred_honest)
cm = confusion_matrix(y_true, pred_honest)

# Neutral statistics
neutral_on_neg = np.sum((y_true == 0) & (pred_3class == 1))
neutral_on_pos = np.sum((y_true == 1) & (pred_3class == 1))
total_neutral = np.sum(neutral_mask)

print("="*60)
print("ZERO-SHOT EVALUATION - BanglaBook")
print("="*60)
print(f"Model: {MODEL_NAME}")
print(f"Test samples: {len(test_df)}")
print(f"\nAccuracy: {acc:.4f} ({acc*100:.2f}%)")
print(f"F1-score: {f1:.4f} ({f1*100:.2f}%)")
print("\nConfusion Matrix [ [TN FP], [FN TP] ]:")
print(cm)
print("\nClassification Report:")
print(classification_report(y_true, pred_honest, target_names=["NEG", "POS"]))
print(f"\nNeutral predictions: {total_neutral} ({total_neutral/len(y_true)*100:.2f}%)")
print(f"  True Negative → Neutral: {neutral_on_neg}")
print(f"  True Positive → Neutral: {neutral_on_pos}")

# Save results
with open(os.path.join(RESULTS_PATH, "banglabook_zero_shot.txt"), "w", encoding="utf-8") as f:
    f.write(f"MODEL: {MODEL_NAME}\n")
    f.write("DATASET: BanglaBook (binary: 0=NEG, 1=POS)\n")
    f.write("EVALUATION: Honest approach (neutral predictions count as wrong)\n\n")
    f.write(f"Test samples: {len(test_df)}\n")
    f.write(f"Accuracy: {acc:.4f}\n")
    f.write(f"F1-score: {f1:.4f}\n\n")
    f.write("Confusion Matrix [ [TN FP], [FN TP] ]:\n")
    f.write(str(cm) + "\n\n")
    f.write("Classification Report:\n")
    f.write(classification_report(y_true, pred_honest, target_names=["NEG", "POS"]))
    f.write(f"\nNeutral predictions: {total_neutral} ({total_neutral/len(y_true)*100:.2f}%)\n")
    f.write(f"  True Negative → Neutral: {neutral_on_neg}\n")
    f.write(f"  True Positive → Neutral: {neutral_on_pos}\n")

print(f"\nResults saved to: {RESULTS_PATH}/banglabook_zero_shot.txt")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

Map:   0%|          | 0/30253 [00:00<?, ? examples/s]

pytorch_model.bin:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-xlm-roberta-base-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

ZERO-SHOT EVALUATION - BanglaBook
Model: cardiffnlp/twitter-xlm-roberta-base-sentiment
Test samples: 30253

Accuracy: 0.5529 (55.29%)
F1-score: 0.7020 (70.20%)

Confusion Matrix [ [TN FP], [FN TP] ]:
[[  790  1145]
 [12382 15936]]

Classification Report:
              precision    recall  f1-score   support

         NEG       0.06      0.41      0.10      1935
         POS       0.93      0.56      0.70     28318

    accuracy                           0.55     30253
   macro avg       0.50      0.49      0.40     30253
weighted avg       0.88      0.55      0.66     30253


Neutral predictions: 10725 (35.45%)
  True Negative → Neutral: 781
  True Positive → Neutral: 9944

Results saved to: /content/drive/MyDrive/XMLROBERT/Dataset_Dimension_Comparison/results/banglabook_zero_shot.txt


**Fine-tune on BanglaBook**

In [14]:
# Cell 6: Fine-tune on BanglaBook
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed
)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

MODEL_NAME = "cardiffnlp/twitter-xlm-roberta-base-sentiment"
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 3
LR = 2e-5
SEED = 42

set_seed(SEED)

# Load binary datasets
train_df = pd.read_csv(os.path.join(PREPROCESSED_PATH, "BanglaBook_train_binary.csv"))
val_df   = pd.read_csv(os.path.join(PREPROCESSED_PATH, "BanglaBook_val_binary.csv"))
test_df  = pd.read_csv(os.path.join(PREPROCESSED_PATH, "BanglaBook_test_binary.csv"))

# Ensure correct column names
train_df = train_df.rename(columns={"text": "text", "label": "label"})
val_df   = val_df.rename(columns={"text": "text", "label": "label"})
test_df  = test_df.rename(columns={"text": "text", "label": "label"})

train_df["text"] = train_df["text"].astype(str)
val_df["text"]   = val_df["text"].astype(str)
test_df["text"]  = test_df["text"].astype(str)

print("Train:", train_df.shape, train_df["label"].value_counts().to_dict())
print("Val  :", val_df.shape,   val_df["label"].value_counts().to_dict())
print("Test :", test_df.shape,  test_df["label"].value_counts().to_dict())

# Convert to HF Dataset
train_ds = Dataset.from_pandas(train_df[["text", "label"]])
val_ds   = Dataset.from_pandas(val_df[["text", "label"]])
test_ds  = Dataset.from_pandas(test_df[["text", "label"]])

# Tokenize
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
collator = DataCollatorWithPadding(tokenizer=tokenizer)

def tok(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

train_ds = train_ds.map(tok, batched=True)
val_ds   = val_ds.map(tok, batched=True)
test_ds  = test_ds.map(tok, batched=True)

train_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])
val_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])

# Load model for binary classification
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    ignore_mismatched_sizes=True
)

# Metrics function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds)
    }

# Training arguments
args = TrainingArguments(
    output_dir=os.path.join(RESULTS_PATH, "xlmr_banglabook_finetuned"),
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    seed=SEED,
    logging_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    compute_metrics=compute_metrics
)

# Train
print("\nStarting training...")
trainer.train()

# Evaluate on test
pred = trainer.predict(test_ds)
logits = pred.predictions
y_pred = np.argmax(logits, axis=1)
y_true = test_df["label"].values

# Detailed metrics
acc = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred)
report = classification_report(y_true, y_pred, target_names=["NEG", "POS"])

tn, fp, fn, tp = cm.ravel()

print("\n" + "="*60)
print("FINE-TUNED EVALUATION RESULTS - BanglaBook")
print("="*60)
print(f"Model: {MODEL_NAME}")
print(f"Test samples: {len(test_df)}")
print(f"\nAccuracy:  {acc:.4f} ({acc*100:.2f}%)")
print(f"Precision: {precision:.4f} ({precision*100:.2f}%)")
print(f"Recall:    {recall:.4f} ({recall*100:.2f}%)")
print(f"F1-score:  {f1:.4f} ({f1*100:.2f}%)")
print("\nConfusion Matrix [ [TN FP], [FN TP] ]:")
print(cm)
print("\nPer-class breakdown:")
print(f"True Negatives (TN): {tn}")
print(f"False Positives (FP): {fp}")
print(f"False Negatives (FN): {fn}")
print(f"True Positives (TP): {tp}")
print("\nClassification Report:")
print(report)

# Save results
with open(os.path.join(RESULTS_PATH, "banglabook_fine_tuned.txt"), "w", encoding="utf-8") as f:
    f.write(f"MODEL: {MODEL_NAME}\n")
    f.write("DATASET: BanglaBook (binary: 0=NEG, 1=POS)\n")
    f.write("EVALUATION: Fine-tuned\n")
    f.write(f"Max_len={MAX_LEN}, Batch={BATCH_SIZE}, Epochs={EPOCHS}, LR={LR}, Seed={SEED}\n\n")
    f.write(f"Test samples: {len(test_df)}\n\n")
    f.write(f"Accuracy:  {acc:.4f}\n")
    f.write(f"Precision: {precision:.4f}\n")
    f.write(f"Recall:    {recall:.4f}\n")
    f.write(f"F1-score:  {f1:.4f}\n\n")
    f.write("Confusion Matrix [ [TN FP], [FN TP] ]:\n")
    f.write(str(cm) + "\n\n")
    f.write("Per-class breakdown:\n")
    f.write(f"True Negatives (TN): {tn}\n")
    f.write(f"False Positives (FP): {fp}\n")
    f.write(f"False Negatives (FN): {fn}\n")
    f.write(f"True Positives (TP): {tp}\n\n")
    f.write("Classification Report:\n")
    f.write(report)

print("\n" + "="*60)
print(f"Results saved to: {RESULTS_PATH}/banglabook_fine_tuned.txt")
print("="*60)

# Save predictions for error analysis
test_df_with_pred = test_df.copy()
test_df_with_pred["predicted"] = y_pred
test_df_with_pred["correct"] = (y_true == y_pred)
test_df_with_pred.to_csv(os.path.join(RESULTS_PATH, "banglabook_finetuned_predictions.csv"), index=False)
print(f"Predictions saved to: {RESULTS_PATH}/banglabook_finetuned_predictions.csv")

Train: (105882, 2) {1: 99110, 0: 6772}
Val  : (15126, 2) {1: 14159, 0: 967}
Test : (30253, 2) {1: 28318, 0: 1935}


Map:   0%|          | 0/105882 [00:00<?, ? examples/s]

Map:   0%|          | 0/15126 [00:00<?, ? examples/s]

Map:   0%|          | 0/30253 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.156812,0.150080,0.962515,0.980262
2,0.106452,0.142144,0.965887,0.981950
3,0.111071,0.151246,0.967209,0.982671


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


FINE-TUNED EVALUATION RESULTS - BanglaBook
Model: cardiffnlp/twitter-xlm-roberta-base-sentiment
Test samples: 30253

Accuracy:  0.9680 (96.80%)
Precision: 0.9733 (97.33%)
Recall:    0.9930 (99.30%)
F1-score:  0.9831 (98.31%)

Confusion Matrix [ [TN FP], [FN TP] ]:
[[ 1165   770]
 [  199 28119]]

Per-class breakdown:
True Negatives (TN): 1165
False Positives (FP): 770
False Negatives (FN): 199
True Positives (TP): 28119

Classification Report:
              precision    recall  f1-score   support

         NEG       0.85      0.60      0.71      1935
         POS       0.97      0.99      0.98     28318

    accuracy                           0.97     30253
   macro avg       0.91      0.80      0.84     30253
weighted avg       0.97      0.97      0.97     30253


Results saved to: /content/drive/MyDrive/XMLROBERT/Dataset_Dimension_Comparison/results/banglabook_fine_tuned.txt
Predictions saved to: /content/drive/MyDrive/XMLROBERT/Dataset_Dimension_Comparison/results/banglabook_finetun

**Detailed dataset analysis**

In [15]:
# Cell 7: Detailed dataset analysis
from datetime import datetime

# Load binary splits (already have them in memory, but reload for clarity)
train = pd.read_csv(os.path.join(PREPROCESSED_PATH, "BanglaBook_train_binary.csv"))
val   = pd.read_csv(os.path.join(PREPROCESSED_PATH, "BanglaBook_val_binary.csv"))
test  = pd.read_csv(os.path.join(PREPROCESSED_PATH, "BanglaBook_test_binary.csv"))

# Rename for consistency
train = train.rename(columns={"text": "text", "label": "label"})
val   = val.rename(columns={"text": "text", "label": "label"})
test  = test.rename(columns={"text": "text", "label": "label"})

# Add text length columns
for df in [train, val, test]:
    df['text_length'] = df['text'].astype(str).apply(len)
    df['word_count'] = df['text'].astype(str).apply(lambda x: len(x.split()))

# Combine all splits
full_df = pd.concat([train, val, test], ignore_index=True)

print("\n" + "="*70)
print("BANGLA BOOK DATASET ANALYSIS")
print("="*70)

print("\n1. DATASET DIMENSIONS:")
print(f"   Total samples: {len(full_df)}")
print(f"   Total features: {len(full_df.columns)}")
print(f"   Features: {list(full_df.columns)}")

print("\n2. SPLIT DISTRIBUTION:")
print(f"   Train: {len(train)} samples ({len(train)/len(full_df)*100:.2f}%)")
print(f"   Val:   {len(val)} samples ({len(val)/len(full_df)*100:.2f}%)")
print(f"   Test:  {len(test)} samples ({len(test)/len(full_df)*100:.2f}%)")

print("\n3. CLASS DISTRIBUTION (0=NEG, 1=POS):")
print("   Overall:")
neg_total = (full_df['label'] == 0).sum()
pos_total = (full_df['label'] == 1).sum()
print(f"      Negative (0): {neg_total} ({neg_total/len(full_df)*100:.2f}%)")
print(f"      Positive (1): {pos_total} ({pos_total/len(full_df)*100:.2f}%)")

print("\n4. TEXT LENGTH STATISTICS (characters):")
print(f"   Min length:  {full_df['text_length'].min()}")
print(f"   Max length:  {full_df['text_length'].max()}")
print(f"   Mean length: {full_df['text_length'].mean():.2f}")
print(f"   Median:      {full_df['text_length'].median():.2f}")
print(f"   Std dev:     {full_df['text_length'].std():.2f}")

print("\n5. WORD COUNT STATISTICS:")
print(f"   Min words:  {full_df['word_count'].min()}")
print(f"   Max words:  {full_df['word_count'].max()}")
print(f"   Mean words: {full_df['word_count'].mean():.2f}")
print(f"   Median:     {full_df['word_count'].median():.2f}")

print("\n6. DATA QUALITY:")
print(f"   Missing values: {full_df.isnull().sum().sum()}")
print(f"   Duplicate texts: {full_df.duplicated(subset=['text']).sum()}")

# Detailed split analysis
print("\n" + "="*70)
print("TRAIN/VAL/TEST SEPARATE ANALYSIS")
print("="*70)

for name, df in [("TRAIN", train), ("VAL", val), ("TEST", test)]:
    print(f"\n{'-'*50}")
    print(f"{name} SET:")
    print(f"{'-'*50}")
    print(f"   Samples: {len(df)}")
    neg_count = (df['label'] == 0).sum()
    pos_count = (df['label'] == 1).sum()
    print(f"\n   Class distribution:")
    print(f"      Negative (0): {neg_count} ({neg_count/len(df)*100:.2f}%)")
    print(f"      Positive (1): {pos_count} ({pos_count/len(df)*100:.2f}%)")
    print(f"\n   Text length (chars):")
    print(f"      Min: {df['text_length'].min()}, Max: {df['text_length'].max()}, Mean: {df['text_length'].mean():.2f}")
    print(f"   Word count:")
    print(f"      Min: {df['word_count'].min()}, Max: {df['word_count'].max()}, Mean: {df['word_count'].mean():.2f}")

# Sample texts from each class (using original data to show original labels)
print("\n" + "="*70)
print("SAMPLE TEXTS FROM EACH CLASS")
print("="*70)

train_orig = pd.read_csv(os.path.join(DATA_PATH, "train.csv"))

for name, df in [("TRAIN", train_orig)]:
    print(f"\n{name} SET:")
    for label_value, label_name in [(0, "NEGATIVE"), (2, "POSITIVE")]:
        samples = df[df['label'] == label_value].sample(min(3, len(df[df['label'] == label_value])), random_state=42)
        print(f"\n   {label_name} (Label {label_value}) examples:")
        for idx, row in samples.iterrows():
            text = str(row['Review'])[:150] + "..." if len(str(row['Review'])) > 150 else str(row['Review'])
            print(f"      - {text}")

# Save analysis to file
with open(os.path.join(RESULTS_PATH, "banglabook_dataset_analysis.txt"), "w", encoding="utf-8") as f:
    f.write("="*70 + "\n")
    f.write("BANGLA BOOK DATASET ANALYSIS REPORT\n")
    f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write("="*70 + "\n\n")
    f.write(f"Total samples: {len(full_df)}\n")
    f.write(f"Train: {len(train)} ({len(train)/len(full_df)*100:.2f}%)\n")
    f.write(f"Val:   {len(val)} ({len(val)/len(full_df)*100:.2f}%)\n")
    f.write(f"Test:  {len(test)} ({len(test)/len(full_df)*100:.2f}%)\n\n")
    f.write(f"Negative (0): {neg_total} ({neg_total/len(full_df)*100:.2f}%)\n")
    f.write(f"Positive (1): {pos_total} ({pos_total/len(full_df)*100:.2f}%)\n\n")
    f.write(f"Text length - Min: {full_df['text_length'].min()}, Max: {full_df['text_length'].max()}, Mean: {full_df['text_length'].mean():.2f}\n")
    f.write(f"Word count  - Min: {full_df['word_count'].min()}, Max: {full_df['word_count'].max()}, Mean: {full_df['word_count'].mean():.2f}\n")
    f.write(f"Duplicate texts: {full_df.duplicated(subset=['text']).sum()}\n")

print("\n" + "="*70)
print(f"Analysis saved to: {RESULTS_PATH}/banglabook_dataset_analysis.txt")
print("="*70)

# Save summary CSV
summary_data = {
    'Split': ['Train', 'Validation', 'Test', 'Overall'],
    'Samples': [len(train), len(val), len(test), len(full_df)],
    'Negative': [(train['label']==0).sum(), (val['label']==0).sum(), (test['label']==0).sum(), neg_total],
    'Positive': [(train['label']==1).sum(), (val['label']==1).sum(), (test['label']==1).sum(), pos_total],
    'Avg_Word_Count': [train['word_count'].mean(), val['word_count'].mean(), test['word_count'].mean(), full_df['word_count'].mean()]
}
summary_df = pd.DataFrame(summary_data)
summary_df.to_csv(os.path.join(RESULTS_PATH, "banglabook_dataset_summary.csv"), index=False)
print(f"Summary CSV saved to: {RESULTS_PATH}/banglabook_dataset_summary.csv")


BANGLA BOOK DATASET ANALYSIS

1. DATASET DIMENSIONS:
   Total samples: 151261
   Total features: 4
   Features: ['text', 'label', 'text_length', 'word_count']

2. SPLIT DISTRIBUTION:
   Train: 105882 samples (70.00%)
   Val:   15126 samples (10.00%)
   Test:  30253 samples (20.00%)

3. CLASS DISTRIBUTION (0=NEG, 1=POS):
   Overall:
      Negative (0): 9674 (6.40%)
      Positive (1): 141587 (93.60%)

4. TEXT LENGTH STATISTICS (characters):
   Min length:  1
   Max length:  267
   Mean length: 107.54
   Median:      58.00
   Std dev:     102.92

5. WORD COUNT STATISTICS:
   Min words:  1
   Max words:  64
   Mean words: 17.53
   Median:     10.00

6. DATA QUALITY:
   Missing values: 0
   Duplicate texts: 56357

TRAIN/VAL/TEST SEPARATE ANALYSIS

--------------------------------------------------
TRAIN SET:
--------------------------------------------------
   Samples: 105882

   Class distribution:
      Negative (0): 6772 (6.40%)
      Positive (1): 99110 (93.60%)

   Text length (char